# 📓 Notebook 22 — AI Evaluation & Observability

> **Module:** AI Engineering · **Estimated time:** 55–70 min · **Difficulty:** Intermediate

You shipped your first AI feature in NB 18–21. You watched the *demo* answers and they looked great. But how do you know it's still working tomorrow? Next month? After someone tweaks a prompt? After OpenAI updates the model?

This notebook is the difference between *hobby AI* and *production AI*. We cover:

- **Golden datasets** and regression tests for prompts.
- **Metrics that work for LLMs** (when there's no single right answer).
- **LLM-as-judge** — using one model to grade another.
- **Tracing** — capturing every call so you can debug what happened at 3 a.m.
- **Cost dashboards** — knowing what you're spending, before the invoice arrives.
- **A/B testing prompts and models** — the discipline behind every production-grade AI feature.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Build a **golden dataset** for an AI feature and re-run it on every change.
2. Choose the right metric: **exact-match**, **structured-output match**, **semantic similarity**, **LLM-as-judge**.
3. Implement **call tracing** so every prompt + response is logged.
4. Compute a **cost dashboard** from a trace log.
5. Run an **A/B test of two prompt variants** and reach a defensible verdict.
6. Detect **regression** (a new prompt makes things worse on previously-passing examples).

## ✅ Prerequisites

NB 18–21 (the AI engineering toolkit), NB 13 (statistics for A/B testing).

## 1. Why "look at it and it seems fine" doesn't scale

Imagine you ship an AI inbox-triage feature. Three months later, someone reports that "complaint" messages are being labelled "feature request". You investigate:

- Was the model updated by the provider? Yes, two weeks ago.
- Did anyone tweak the prompt? Yes, last week.
- Which one caused the regression? You have no idea — there's no log.

The cure for this is **observability**: log everything, evaluate against a fixed golden set on every change, and chart cost / latency / accuracy over time. None of these are technically hard. They take *discipline*.

## 2. Setup — the MockLLM and a tiny golden dataset

In [ ]:
import json
import re
import time
import hashlib
from collections import Counter
from dataclasses import dataclass, asdict, field
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})


class MockLLM:
    """Same offline mock pattern used throughout the course."""
    POSITIVE = {"love", "great", "amazing", "excellent", "happy", "fantastic",
                "perfect", "delight", "smooth", "fast", "thank", "good"}
    NEGATIVE = {"hate", "terrible", "awful", "bad", "slow", "broken", "bug",
                "crash", "angry", "frustrating", "refund", "cancel", "worst"}
    BILLING  = {"refund", "invoice", "charge", "bill", "subscription", "renew", "payment", "price"}
    TECH     = {"error", "crash", "bug", "broken", "login", "password", "loading", "freeze", "stuck"}
    FEATURE  = {"feature", "request", "missing", "wish", "add", "support"}

    def _kw_score(self, text, vocab):
        return sum(1 for w in re.findall(r"[a-z']+", text.lower()) if w in vocab)

    def chat(self, messages, temperature=0.0, model="mock-mini"):
        sys_msg = next((m["content"] for m in messages if m["role"] == "system"), "").lower()
        user_msgs = [m["content"] for m in messages if m["role"] == "user"]
        user_msg = user_msgs[-1] if user_msgs else ""

        out = "..."
        if "json" in sys_msg and "sentiment" in sys_msg:
            pos = self._kw_score(user_msg, self.POSITIVE)
            neg = self._kw_score(user_msg, self.NEGATIVE)
            sentiment = "positive" if pos > neg else "negative" if neg > pos else "neutral"
            topic_scores = {
                "billing": self._kw_score(user_msg, self.BILLING),
                "tech":    self._kw_score(user_msg, self.TECH),
                "feature": self._kw_score(user_msg, self.FEATURE),
            }
            topic = max(topic_scores, key=topic_scores.get) if max(topic_scores.values()) > 0 else "other"
            out = json.dumps({"sentiment": sentiment, "topic": topic})
        elif "sentiment" in sys_msg:
            pos = self._kw_score(user_msg, self.POSITIVE)
            neg = self._kw_score(user_msg, self.NEGATIVE)
            out = "positive" if pos > neg else "negative" if neg > pos else "neutral"

        # Rough token-count proxy
        tokens_in  = sum(len(m["content"]) for m in messages) // 4
        tokens_out = len(out) // 4
        return {"text": out, "model": model,
                "tokens_in": tokens_in, "tokens_out": tokens_out,
                "latency_s": 0.001 + 0.0001 * tokens_out}


llm = MockLLM()
print("MockLLM ready ✅")


> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. Once you want real intelligence in the answers, swap one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else.
> See the [LLM Providers Guide](./A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


### Build the golden dataset — 12 labelled examples

In [ ]:
# Each row: (input text, expected sentiment, expected topic)
GOLDEN = [
    ("I love the new dashboard! It's so much faster.",        "positive", "feature"),
    ("The app crashed three times today. Very frustrated.",   "negative", "tech"),
    ("Could you please add CSV export?",                       "neutral",  "feature"),
    ("My invoice doesn't match my plan, please refund.",       "negative", "billing"),
    ("Everything works fine, no complaints.",                  "neutral",  "other"),
    ("Login is broken after the latest update.",               "negative", "tech"),
    ("Fantastic support! Resolved my issue in minutes.",       "positive", "other"),
    ("Why is the renewal price so high? Considering cancel.",  "negative", "billing"),
    ("New feature is great but loading is slow on mobile.",    "neutral",  "tech"),
    ("Please add support for Markdown in the editor.",         "neutral",  "feature"),
    ("Thanks, the team is amazing!",                           "positive", "other"),
    ("Charged twice for my subscription last month.",          "negative", "billing"),
]

golden_df = pd.DataFrame(GOLDEN, columns=["text", "expected_sentiment", "expected_topic"])
print(f"Golden dataset: {len(golden_df)} labelled examples")
print(golden_df.head())


> 💡 **The golden dataset is the single most important AI-evaluation artefact.** Build it before you ship; grow it whenever a real user reports a bad output. After a year you'll have a 500-row regression test that catches almost every "did anything change?" question.

## 3. Run the feature and score it

In [ ]:
SYSTEM = ("You analyse customer messages. Return JSON with keys "
          "'sentiment' (positive/neutral/negative) and 'topic' (billing/tech/feature/other). "
          "Reply with the JSON only.")

def analyse(text):
    r = llm.chat(messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": text},
    ])
    try:
        labels = json.loads(r["text"])
    except json.JSONDecodeError:
        labels = {"sentiment": "unknown", "topic": "unknown"}
    labels["_meta"] = {"tokens_in": r["tokens_in"], "tokens_out": r["tokens_out"],
                       "latency_s": r["latency_s"], "model": r["model"]}
    return labels


# Score the golden set
rows = []
for _, row in golden_df.iterrows():
    pred = analyse(row["text"])
    rows.append({
        "text":              row["text"],
        "expected_sentiment": row["expected_sentiment"],
        "expected_topic":     row["expected_topic"],
        "predicted_sentiment": pred["sentiment"],
        "predicted_topic":     pred["topic"],
        "tokens_in":  pred["_meta"]["tokens_in"],
        "tokens_out": pred["_meta"]["tokens_out"],
        "latency_s":  pred["_meta"]["latency_s"],
    })
eval_df = pd.DataFrame(rows)
eval_df["sentiment_ok"] = eval_df["expected_sentiment"] == eval_df["predicted_sentiment"]
eval_df["topic_ok"]     = eval_df["expected_topic"]     == eval_df["predicted_topic"]
eval_df["both_ok"]      = eval_df["sentiment_ok"] & eval_df["topic_ok"]

print(eval_df[["text", "expected_sentiment", "predicted_sentiment",
                "expected_topic", "predicted_topic",
                "sentiment_ok", "topic_ok"]].to_string(index=False))


In [ ]:
# Headline metrics
n = len(eval_df)
print(f"\nGolden-set results ({n} examples):")
print(f"  Sentiment accuracy : {eval_df['sentiment_ok'].mean():.0%}")
print(f"  Topic accuracy     : {eval_df['topic_ok'].mean():.0%}")
print(f"  Both correct       : {eval_df['both_ok'].mean():.0%}")


**That's an AI eval in 30 lines of code.** Run it every time you change a prompt; if `both_ok` goes down by more than a few points, *don't ship*.

## 4. Per-class metrics — where exactly is it failing?

In [ ]:
# Confusion-matrix-style table for sentiment
ct = pd.crosstab(eval_df["expected_sentiment"], eval_df["predicted_sentiment"],
                 margins=True, margins_name="total")
print("Sentiment — expected (rows) vs predicted (cols):")
print(ct)
print()
ct = pd.crosstab(eval_df["expected_topic"], eval_df["predicted_topic"],
                 margins=True, margins_name="total")
print("Topic — expected (rows) vs predicted (cols):")
print(ct)


**Reading the crosstabs:**

- Diagonal cells = correct.
- Off-diagonal cells = the most informative bit. If you see many "feature → other" errors, *that* is the bucket to focus on improving the prompt.

## 5. LLM-as-judge — when there's no single right answer

Open-ended outputs (summaries, generated emails, code suggestions) have no exact match. The standard trick: ask *another* LLM to grade them.

```
   Input text                  ──┐
                                  │
   Reference / criteria        ──┤
                                  ├──►  judge LLM  ──►  score (1-5) + reasoning
   Candidate output            ──┘
```

This sounds circular but works surprisingly well in practice. Use it for fuzzy axes (helpfulness, tone, completeness) where you can't write a rule.

In [ ]:
def llm_as_judge(input_text: str, candidate: str, criteria: str) -> dict:
    """Score a candidate output 1-5 with a brief reason. (Mock — uses keyword heuristics.)"""
    # Real implementation: another llm.chat call with a careful system prompt.
    # Mock: score based on keyword presence (1 if matches, 5 if matches strongly).
    user_kws = set(re.findall(r"[a-z']+", input_text.lower()))
    cand_kws = set(re.findall(r"[a-z']+", candidate.lower()))
    overlap = len(user_kws & cand_kws)
    if overlap == 0:
        score, reason = 1, "candidate shares no vocabulary with input"
    elif overlap < 3:
        score, reason = 3, "candidate shares some content but not deep enough"
    else:
        score, reason = 5, "candidate shares substantial content with input"
    return {"score": score, "reason": reason, "criteria": criteria}


# Demo
out = llm_as_judge(
    input_text = "Refunds are issued within 5 business days. Customers can request a refund...",
    candidate  = "You'll get your refund in about a week through the Billing page.",
    criteria   = "Faithful to the source and easy to understand."
)
print(out)


> ⚠️ **LLM-as-judge has known biases.** Models tend to prefer their own outputs, longer answers, and confidently-worded text. Mitigations:
>
> 1. Use a *different* (ideally smaller) model as the judge.
> 2. Calibrate by asking the judge to score 20 hand-rated examples and compare.
> 3. Use it as a *signal*, not the sole gate. Pair with golden-set checks.

## 6. Tracing — log every call so you can debug 3 a.m. failures

In [ ]:
@dataclass
class TraceEvent:
    """One record per LLM call."""
    timestamp:    float
    request_id:   str
    feature:      str
    model:        str
    prompt_hash:  str
    user_input:   str
    output:       str
    tokens_in:    int
    tokens_out:   int
    latency_s:    float
    error:        str | None = None


def trace_call(feature: str, user_input: str, system_prompt: str) -> TraceEvent:
    """Wrap an llm.chat call and record everything."""
    request_id = hashlib.sha1(f"{time.time()}{user_input}".encode()).hexdigest()[:10]
    prompt_hash = hashlib.sha1(system_prompt.encode()).hexdigest()[:10]
    try:
        r = llm.chat(messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_input},
        ])
        return TraceEvent(
            timestamp=time.time(), request_id=request_id, feature=feature,
            model=r["model"], prompt_hash=prompt_hash,
            user_input=user_input, output=r["text"],
            tokens_in=r["tokens_in"], tokens_out=r["tokens_out"],
            latency_s=r["latency_s"],
        )
    except Exception as e:
        return TraceEvent(timestamp=time.time(), request_id=request_id, feature=feature,
                          model="error", prompt_hash=prompt_hash,
                          user_input=user_input, output="",
                          tokens_in=0, tokens_out=0, latency_s=0,
                          error=str(e))


# Run a small batch and collect traces
TRACE = []
for text in golden_df["text"].tolist() * 3:    # run 3x to get 36 events
    TRACE.append(trace_call("inbox_triage", text, SYSTEM))

trace_df = pd.DataFrame([asdict(t) for t in TRACE])
print(f"Logged {len(trace_df)} events.")
print(trace_df[["timestamp", "request_id", "feature", "tokens_in", "tokens_out", "latency_s"]].head())


**What goes in every trace row:**

- `timestamp` and `request_id` — so you can correlate with logs from other systems.
- `feature` and `prompt_hash` — track which feature / prompt version was used.
- `user_input` and `output` — for after-the-fact inspection.
- `tokens_in`, `tokens_out`, `latency_s` — the cost / SLO numbers.
- `error` — the difference between a successful run and one that fell through to a fallback.

> 🎯 In production, this is what tools like **LangSmith**, **Helicone**, **Phoenix**, and **Arize** automate. The conceptual structure is the same — they're a hosted version of `trace_df`.

## 7. The cost dashboard

Once you have a trace, the cost dashboard is just pandas + matplotlib.

In [ ]:
# Pricing (per 1K tokens) — change once, propagate everywhere
PRICE_IN_PER_1K  = 0.0006
PRICE_OUT_PER_1K = 0.0024

trace_df["cost_usd"] = (trace_df["tokens_in"] / 1000 * PRICE_IN_PER_1K +
                         trace_df["tokens_out"] / 1000 * PRICE_OUT_PER_1K)
trace_df["dt"] = pd.to_datetime(trace_df["timestamp"], unit="s")
trace_df = trace_df.sort_values("timestamp").reset_index(drop=True)
trace_df["request_index"] = trace_df.index

# Summary numbers
print(f"Total calls           : {len(trace_df)}")
print(f"Total cost            : ${trace_df['cost_usd'].sum():.5f}")
print(f"Mean cost per call    : ${trace_df['cost_usd'].mean():.5f}")
print(f"Mean latency          : {trace_df['latency_s'].mean()*1000:.1f} ms")
print(f"p95 latency           : {trace_df['latency_s'].quantile(0.95)*1000:.1f} ms")
print(f"Error rate            : {(trace_df['error'].notna()).mean():.1%}")


In [ ]:
# Mini dashboard
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# 1) Latency distribution
ax = axes[0]
ax.hist(trace_df["latency_s"] * 1000, bins=15, color="#4C72B0", edgecolor="black", alpha=0.85)
ax.axvline(trace_df["latency_s"].quantile(0.95) * 1000, color="red", ls="--",
            label=f"p95 = {trace_df['latency_s'].quantile(0.95)*1000:.1f} ms")
ax.set_title("Latency distribution")
ax.set_xlabel("ms"); ax.legend()

# 2) Cost cumulative
ax = axes[1]
ax.plot(trace_df["request_index"], trace_df["cost_usd"].cumsum() * 1000,
         lw=2, color="#55A467")
ax.set_title("Cumulative spend")
ax.set_xlabel("Request number"); ax.set_ylabel("Total cost (milli-$)")

# 3) Tokens per call
ax = axes[2]
ax.scatter(trace_df["tokens_in"], trace_df["tokens_out"], alpha=0.7, color="#DD8452",
            edgecolor="black")
ax.set_title("Tokens in vs out")
ax.set_xlabel("Tokens in"); ax.set_ylabel("Tokens out")

plt.tight_layout(); plt.show()


**You now have, in 50 lines of code, what most teams pay $300/month per seat for.** The hosted observability tools add real-time pipelines, alerts, and a UI — but the underlying data model is exactly the `trace_df` above.

## 8. A/B-testing two prompt variants

You want to know whether tweaking the prompt improved things. Run both prompts against the same golden set and compare honestly.

In [ ]:
SYSTEM_V1 = SYSTEM      # the baseline
SYSTEM_V2 = SYSTEM + " Be especially careful about distinguishing 'feature requests' from 'feedback'."

def run_variant(variant_name, system_prompt, golden_df):
    rows = []
    for _, row in golden_df.iterrows():
        r = llm.chat(messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": row["text"]},
        ])
        try:
            pred = json.loads(r["text"])
        except json.JSONDecodeError:
            pred = {"sentiment": "unknown", "topic": "unknown"}
        rows.append({
            "variant": variant_name,
            "sentiment_ok": pred.get("sentiment") == row["expected_sentiment"],
            "topic_ok":     pred.get("topic")     == row["expected_topic"],
            "tokens_in":    r["tokens_in"],
            "tokens_out":   r["tokens_out"],
            "latency_s":    r["latency_s"],
        })
    return pd.DataFrame(rows)

results = pd.concat([
    run_variant("v1_baseline", SYSTEM_V1, golden_df),
    run_variant("v2_with_hint", SYSTEM_V2, golden_df),
])

summary = (results.groupby("variant")
                  .agg(n=("sentiment_ok", "size"),
                       sentiment_acc=("sentiment_ok", "mean"),
                       topic_acc=("topic_ok", "mean"),
                       mean_tokens_out=("tokens_out", "mean"),
                       p95_latency_ms=("latency_s", lambda s: s.quantile(0.95)*1000))
                  .round(3))
print(summary)


**How to read this:**

- The accuracy columns answer "did v2 help?".
- The tokens / latency columns answer "did v2 hurt us elsewhere?".
- For tiny golden sets you also need a confidence interval — NB 13 has the formulas.

> 💡 **The discipline.** Never ship a new prompt without running the eval set on both versions and checking *all four* numbers: accuracy ↑, tokens →, latency →, cost →.

## 9. Regression detection — did a previously-passing case break?

In [ ]:
# Compare v1 and v2 example-by-example
side_by_side = pd.merge(
    results[results["variant"]=="v1_baseline"].reset_index().rename(columns={"index":"i"}),
    results[results["variant"]=="v2_with_hint"].reset_index().rename(columns={"index":"i"}),
    on="i", suffixes=("_v1", "_v2"),
)

# A regression: v1 was right, v2 is wrong
side_by_side["sentiment_regression"] = side_by_side["sentiment_ok_v1"] & ~side_by_side["sentiment_ok_v2"]
side_by_side["topic_regression"]     = side_by_side["topic_ok_v1"]     & ~side_by_side["topic_ok_v2"]

n_regressions = (side_by_side["sentiment_regression"] | side_by_side["topic_regression"]).sum()
print(f"New prompt broke {n_regressions} previously-passing examples out of {len(side_by_side)}.")


> 🎯 **Why regression detection matters more than headline accuracy.** A new prompt that's 1 point more accurate *on average* but breaks an *important* case (e.g., a VIP customer) can be worse than the baseline. Always look at example-level deltas before shipping.

## 10. Putting it together — the eval pipeline

```
   on every change:
     1. Run the golden set
     2. Compute accuracy, cost, latency, error rate
     3. Compare against the *previous* baseline (stored in version control)
     4. Refuse to ship if accuracy drops more than X%
                       OR cost rises more than Y%
                       OR p95 latency rises more than Z%
                       OR any "important" example regresses
```

This isn't fancy — it's the same shape as a Python `pytest` suite, applied to a probabilistic model. Bake it into CI and the AI feature becomes as boring (= reliable) as any other code.

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add a per-class F1 metric

Use `sklearn.metrics.f1_score(..., average=None, labels=[...])` to compute the F1 score per topic class. Print which topic the model is *worst* at.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
from sklearn.metrics import f1_score
labels = ["billing", "tech", "feature", "other"]
f1 = f1_score(eval_df["expected_topic"], eval_df["predicted_topic"],
              labels=labels, average=None, zero_division=0)
worst_idx = int(np.argmin(f1))
print(pd.Series(f1, index=labels).round(3).to_string())
print(f"\nWorst topic: {labels[worst_idx]} (F1 = {f1[worst_idx]:.3f})")
```

Per-class F1 is the right metric for imbalanced multi-class problems. The worst class tells you exactly which examples to focus on improving.
</details>

### Exercise 2 — ⭐⭐ Trace-based cost forecast

Using `trace_df`, forecast the **monthly cost** if today's traffic continues at the same rate. Assume each request in the trace represents one *minute* of production traffic.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
mean_cost_per_call = trace_df["cost_usd"].mean()
calls_per_minute = 1
calls_per_month  = calls_per_minute * 60 * 24 * 30

monthly_cost = mean_cost_per_call * calls_per_month
print(f"Mean cost / call: ${mean_cost_per_call:.5f}")
print(f"Calls / month  : {calls_per_month:,}")
print(f"Forecast cost  : ${monthly_cost:,.2f}/month")
```

A simple cost forecast like this — done *before* you scale up traffic — is what stops the dreaded "we 10×'d our load and the API bill 10×'d too" surprise.
</details>

### Exercise 3 — ⭐⭐ Add error-rate to the dashboard

Modify the dashboard to add a **4th panel** showing error rate over time (errors divided by total calls in a sliding window of 10 calls). Use the `trace_df` column `error`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
trace_df["is_error"] = trace_df["error"].notna()
trace_df["error_rate_w10"] = trace_df["is_error"].rolling(10, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(trace_df["request_index"], trace_df["error_rate_w10"] * 100,
         lw=2, color="#C44E52")
ax.set_xlabel("Request number")
ax.set_ylabel("Error rate (%) — 10-call rolling")
ax.set_title("Error rate over time")
plt.tight_layout(); plt.show()
```

Note: in this mock notebook the error rate is always 0 because the mock never fails. In production this chart is the canary that tells you "something just changed".
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — ⭐⭐⭐ Per-feature daily cost trend

Add a `feature` column to the trace before saving. Aggregate by feature, simulating 3 features each with different volumes, and print a daily cost-per-feature summary.


<details>
<summary>💡 <b>Solution</b></summary>

```python
# Simulate three features being served from the same LLM
np.random.seed(0)
trace_df["feature"] = np.random.choice(
    ["inbox_triage", "auto_summary", "doc_extract"], size=len(trace_df),
    p=[0.6, 0.25, 0.15]
)
by_feature = trace_df.groupby("feature").agg(
    n_calls=("request_id", "count"),
    tokens=("tokens_in", "sum"),
    cost=("cost_usd", "sum"),
    mean_latency_ms=("latency_s", lambda s: s.mean() * 1000),
).round(3)
print(by_feature)
```

**Feature-level cost dashboards** are how you prevent the *"one
feature is silently 80% of our LLM bill"* surprise. Always tag
each call with which feature/product it came from, *before* the
data lands in your trace store.

</details>

### Stretch exercise B — ⭐⭐⭐ Pre-commit eval that compares two prompts

Modify `pre_commit_check` to compare *two* prompt variants against the golden set and *only* allow the new one to merge if it improves accuracy on the golden set without losing >5pts on any per-class metric.


<details>
<summary>💡 <b>Solution</b></summary>

```python
def safer_pre_commit_check(old_prompt, new_prompt, golden_df):
    old_res = run_variant("old", old_prompt, golden_df)
    new_res = run_variant("new", new_prompt, golden_df)

    metrics = ["sentiment_ok", "topic_ok"]
    old_acc = old_res[metrics].mean()
    new_acc = new_res[metrics].mean()

    print(f"           old   new   Δ")
    for m in metrics:
        print(f"  {m:<12}{old_acc[m]:.2f}  {new_acc[m]:.2f}  {new_acc[m]-old_acc[m]:+.2f}")

    if (new_acc < old_acc - 0.05).any():
        print("❌ Regression on at least one metric — refusing the merge.")
        return False
    print("✅ New prompt at least as good as old on every metric.")
    return True


safer_pre_commit_check(SYSTEM_V1, SYSTEM_V2, golden_df)
```

**Pre-commit *comparison* checks beat absolute baselines.** A new
prompt might genuinely improve the model overall *while* breaking
the one example a VIP customer cares about. Comparing
*per-metric* (or even per-example) catches that.

</details>

### Stretch exercise C — ⭐⭐⭐ Exact-match vs case-insensitive vs semantic

Score the same model output against three different correctness metrics and discuss when each is appropriate.

```python
expected = "Refunds are issued within 5 business days."
actual   = "refunds are issued within five business days"
```

Compute:
1. Exact match
2. Case-insensitive substring match
3. Token-overlap Jaccard

Print all three.

In [ ]:
# Your code here  👇
import re
expected = "Refunds are issued within 5 business days."
actual   = "refunds are issued within five business days"

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import re
expected = "Refunds are issued within 5 business days."
actual   = "refunds are issued within five business days"

exact = (expected == actual)
ci_substring = expected.strip(".").lower() in actual.lower()

def toks(s): return set(re.findall(r"[a-z0-9]+", s.lower()))
j = len(toks(expected) & toks(actual)) / max(1, len(toks(expected) | toks(actual)))

print(f"exact:                  {exact}")
print(f"case-insensitive sub:   {ci_substring}")
print(f"Jaccard token overlap:  {j:.2f}")
```

**Reasoning.** Three metrics, three failure modes. (1) **Exact match** is the strictest — it fails here because of '5' vs 'five' and the missing period. Use it only when output is structured and you control the format. (2) **Case-insensitive substring** is forgiving of casing and punctuation but won't catch 5↔five. Reasonable for sentence-level recall checks. (3) **Token overlap (Jaccard)** is forgiving of word order and small wording changes — but it treats every word as equal, including 'the'. For real eval pipelines, the gold standard is a combination: exact-match on a held-out structured field, plus an LLM-as-judge on the free-text portion.
</details>

### Stretch exercise D — ⭐⭐⭐ Detect a regression between two prompts

You changed your system prompt and want to be sure it didn't break the answers on your golden dataset. Run both prompts through the MockLLM and report which queries got a **different** answer.

Use any small dataset of 5 messages.

In [ ]:
# Your code here  👇
QUERIES = [
    "My invoice is wrong",
    "App crashes on startup",
    "Add a dark mode please",
    "Where is my refund",
    "Can I export my data?",
]
PROMPT_A = "Classify the topic into one of: billing, tech, feature."
PROMPT_B = "Reply with the topic in JSON: {sentiment, topic}. topic ∈ billing/tech/feature."

# Run both prompts through `llm` and report differences.
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
QUERIES = [
    "My invoice is wrong",
    "App crashes on startup",
    "Add a dark mode please",
    "Where is my refund",
    "Can I export my data?",
]
PROMPT_A = "Classify the topic into one of: billing, tech, feature."
PROMPT_B = "Reply with the topic in JSON: {sentiment, topic}. topic ∈ billing/tech/feature."

def run(prompt, queries):
    return [
        llm.chat(messages=[
            {"role": "system", "content": prompt},
            {"role": "user",   "content": q},
        ])["text"]
        for q in queries
    ]

a_results = run(PROMPT_A, QUERIES)
b_results = run(PROMPT_B, QUERIES)

for q, a, b in zip(QUERIES, a_results, b_results):
    mark = "DIFF" if a != b else "same"
    print(f"  [{mark}] {q!r}\n    A: {a!r}\n    B: {b!r}")
```

**Reasoning.** Three habits this builds. (1) **Compare against your own previous version, not just against a 'truth'.** Golden datasets drift; even a 'right' answer changes shape when you change the prompt format (free text vs JSON). Detecting *change* and then asking 'was the change OK?' is more honest than chasing absolute correctness. (2) **Tag, don't filter.** Print everything with a 'DIFF' / 'same' marker; resist the urge to only print the diffs — context matters. (3) For larger datasets, version both prompts in git, save the outputs as JSONL, and diff them — production eval pipelines like Braintrust or PromptLayer just automate this.
</details>

## 🎁 Bonus mini-project — A pre-commit eval check

Write a function `pre_commit_check(prompt: str, golden_df, baseline_acc: float) -> bool` that:

1. Runs the prompt against the golden set.
2. Computes overall accuracy.
3. Returns `True` if accuracy ≥ `baseline_acc - 0.05` (i.e., didn't regress by more than 5 points).
4. Otherwise returns `False` and prints which examples regressed.

This is the function you'd run in a CI hook before merging a prompt change.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def pre_commit_check(prompt: str, golden_df, baseline_acc: float,
                     tolerance: float = 0.05) -> bool:
    rows = []
    for _, row in golden_df.iterrows():
        r = llm.chat(messages=[
            {"role": "system", "content": prompt},
            {"role": "user",   "content": row["text"]},
        ])
        try:
            pred = json.loads(r["text"])
        except json.JSONDecodeError:
            pred = {"sentiment": "unknown", "topic": "unknown"}
        rows.append({
            "text":          row["text"],
            "expected":      (row["expected_sentiment"], row["expected_topic"]),
            "predicted":     (pred.get("sentiment"), pred.get("topic")),
            "ok": pred.get("sentiment") == row["expected_sentiment"]
                  and pred.get("topic") == row["expected_topic"],
        })
    res = pd.DataFrame(rows)
    acc = res["ok"].mean()
    print(f"Golden-set accuracy : {acc:.0%}  (baseline {baseline_acc:.0%}, tolerance {tolerance:.0%})")
    if acc < baseline_acc - tolerance:
        regressed = res[~res["ok"]]
        print(f"❌ Regression detected on {len(regressed)} examples:")
        for _, r in regressed.iterrows():
            print(f"  - {r['text'][:60]!r:<62} expected={r['expected']}, got={r['predicted']}")
        return False
    print("✅ Eval passed.")
    return True


# Test with current baseline
pre_commit_check(SYSTEM, golden_df, baseline_acc=0.75)
```

**Wire this into CI** (`.github/workflows/eval.yml`) and a prompt change can never silently regress production. The function exits non-zero, the merge gets blocked, and someone investigates *before* the customer notices.
</details>

## 🧠 Key takeaways

1. **A golden dataset is the most important AI artefact.** Build it before you ship.
2. **Match metrics to your task**: exact-match for classification, semantic similarity / LLM-as-judge for free-form.
3. **LLM-as-judge has biases.** Calibrate it; use it as a signal, not the sole gate.
4. **Trace every call** — timestamp, request_id, prompt_hash, tokens, latency, error. The data is small and the diagnostic value is huge.
5. **A cost dashboard is 30 lines of pandas + matplotlib** on top of the trace.
6. **A/B test prompts** with the same discipline as you'd A/B-test ML models (NB 13 / NB 16).
7. **Watch for regressions example-by-example**, not just headline accuracy.
8. Bake the eval into CI so a bad prompt cannot reach production.

## ✅ Self-assessment

- [ ] Build a 10-example golden dataset for an AI feature
- [ ] Compute accuracy + per-class F1 on the golden set
- [ ] Explain when LLM-as-judge is the right metric (and its biases)
- [ ] Capture a trace record for each LLM call
- [ ] Compute total cost, mean latency, p95 latency, error rate from the trace
- [ ] Run an A/B test of two prompt variants and pick a defensible winner
- [ ] Detect example-level regression between two prompt versions

## 🚀 Next step

Continue with **Module 6 — Production** (`../06_production/23_from_notebook_to_project.ipynb`), where the eval pipeline you just built becomes part of a packaged, tested, scheduled Python project.